# 1일차 7·8교시 · 데이터 품질 검사

유방암 데이터의 일부 특성에 **일부러 오류를 섞은 데이터**를 만들고, 정합성 검사로 문제를 찾아냅니다. 8교시 과제 A(검수 기준서)는 이 노트북의 결과로 작성합니다.

섞어 둔 오류는 모두 **6종류**입니다. 무엇인지 미리 보지 말고 검사로 찾아보세요.

## 1. 실습 데이터 만들기

이 셀은 실행만 하세요. 내용을 읽으면 정답이 보입니다.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer

def make_dirty_data(seed=0):
    rng = np.random.RandomState(seed)
    raw = load_breast_cancer(as_frame=True)
    cols = {"mean radius": "radius", "mean texture": "texture", "mean area": "area",
            "mean smoothness": "smoothness"}
    df = raw.frame[list(cols) + ["target"]].rename(columns={**cols, "target": "label"})
    df.insert(0, "patient_id", [f"P{i:04d}" for i in range(len(df))])
    df.insert(1, "hospital", rng.choice(["A", "B", "C"], size=len(df), p=[0.5, 0.3, 0.2]))
    df.loc[rng.choice(df.index, 12, replace=False), "texture"] = np.nan
    df.loc[rng.choice(df.index, 45, replace=False), "smoothness"] = np.nan
    df.loc[rng.choice(df.index, 4, replace=False), "radius"] = [-3.0, 0.0, 250.0, 999.0]
    df.loc[rng.choice(df.index, 5, replace=False), "hospital"] = ["D", "a", "b", "", "Z"]
    big = df.index[(df["radius"] > 20) & (df["radius"] < 30) & (df["label"] == 0)]
    df.loc[rng.choice(big, 6, replace=False), "label"] = 1
    df = pd.concat([df, df.sample(8, random_state=seed)], ignore_index=True)
    df = df.sample(frac=1, random_state=seed).reset_index(drop=True)
    ids = np.array(df["patient_id"].unique(), dtype=object)
    rng.shuffle(ids)
    test_ids = set(ids[:140])
    train_ids = set(ids[140:]) | set(list(test_ids)[:7])
    return df, train_ids, test_ids

df, train_ids, test_ids = make_dirty_data()
df.head()

## 2. 데이터 명세

| 컬럼 | 자료형 | 허용 범위 / 값 | 설명 |
| --- | --- | --- | --- |
| patient_id | 문자열 | 고유값 | 환자 ID |
| hospital | 문자열 | A, B, C | 검사 병원 |
| radius | 실수 | 5 ~ 30 | 종양 평균 반지름 |
| texture | 실수 | 5 ~ 45 | 질감 |
| area | 실수 | 100 ~ 2,600 | 면적 |
| smoothness | 실수 | 0.04 ~ 0.2 | 매끄러움 |
| label | 정수 | 0(악성), 1(양성종양) | 진단 결과 |

In [ ]:
SPEC = {
    "columns": ["patient_id", "hospital", "radius", "texture", "area", "smoothness", "label"],
    "ranges": {"radius": (5, 30), "texture": (5, 45), "area": (100, 2600), "smoothness": (0.04, 0.2)},
    "categories": {"hospital": {"A", "B", "C"}, "label": {0, 1}},
}

## 3. 검사 ① 스키마

In [ ]:
print("컬럼 일치:", list(df.columns) == SPEC["columns"])
df.dtypes

## 4. 검사 ② 결측

In [ ]:
df.isna().mean().round(3).to_frame("결측률")

## 5. 검사 ③ 범위

In [ ]:
out_of_range = {}
for col, (low, high) in SPEC["ranges"].items():
    bad = df[col].notna() & ~df[col].between(low, high)
    out_of_range[col] = int(bad.sum())
print(out_of_range)
df[df["radius"].notna() & ~df["radius"].between(5, 30)]

## 6. 검사 ④ 범주값

In [ ]:
for col, allowed in SPEC["categories"].items():
    bad = ~df[col].isin(allowed)
    print(col, "허용되지 않은 값:", df.loc[bad, col].unique().tolist(), f"({int(bad.sum())}건)")

## 7. 검사 ⑤ 중복

In [ ]:
print("완전 중복 행:", int(df.duplicated().sum()))
print("ID 중복:", int(df["patient_id"].duplicated().sum()))

## 8. 검사 ⑥ 라벨

라벨 비율을 보고, **도메인 규칙**으로 의심스러운 라벨을 찾습니다. 여기서는 설명을 위해 "반지름이 20 이상인데 양성종양(1)"을 의심 규칙으로 씁니다. 실제로는 전문가가 규칙을 정하고, 의심 건은 원본을 확인합니다.

In [ ]:
print(df["label"].value_counts(normalize=True).round(3))
suspicious = df[(df["radius"] >= 20) & (df["radius"] <= 30) & (df["label"] == 1)]
print("의심 라벨:", len(suspicious), "건")
suspicious

## 9. 검사 ⑦ 학습·테스트 분할 누수

In [ ]:
overlap = train_ids & test_ids
print("학습·테스트에 동시에 있는 환자 수:", len(overlap))

## 10. 검사 결과를 한 표로 모으기

검수 기준(허용 기준)을 먼저 정해 두고 결과와 비교합니다. 기준 숫자는 예시입니다.

In [ ]:
checks = [
    ("스키마", "컬럼 이름·순서 일치", "일치", list(df.columns) == SPEC["columns"]),
    ("결측", "컬럼별 결측률 최댓값", "0.05 이하", round(df.isna().mean().max(), 3)),
    ("범위", "범위 밖 값 건수", "0건", sum(out_of_range.values())),
    ("범주값", "허용되지 않은 hospital 값", "0건", int((~df["hospital"].isin(SPEC["categories"]["hospital"])).sum())),
    ("중복", "완전 중복 행", "0건", int(df.duplicated().sum())),
    ("라벨", "규칙 위반 의심 라벨", "0건", len(suspicious)),
    ("분할 누수", "학습·테스트 ID 교집합", "0건", len(overlap)),
]
def judge(rule, value):
    if rule == "일치":
        return "통과" if value else "실패"
    if rule == "0.05 이하":
        return "통과" if value <= 0.05 else "실패"
    return "통과" if value == 0 else "실패"

pd.DataFrame([{"항목": a, "검사": b, "허용 기준": c, "결과": d, "판정": judge(c, d)} for a, b, c, d in checks])

## 11. 과제 A (8교시)

위 결과를 보고 `day1/templates/data-quality-checklist.md` 양식으로 검수 기준서를 작성합니다.

1. 각 항목의 **허용 기준**이 적절한지 조별로 토론하고 조정하세요.
2. 기준을 벗어난 항목마다 **조치**(삭제, 대체, 원본 확인 요청)를 정하세요.
3. 이 데이터로 학습을 진행해도 되는지 **종합 판정**하세요.

## 12. [선택] 조치 후 다시 검사하기

아래 셀에 정제 코드를 작성하고, 10번 셀을 다시 실행해 모든 항목이 통과하는지 확인해 보세요.

In [ ]:
clean = df.copy()
# 예: clean = clean.drop_duplicates()
# 여기에 정제 코드를 작성하세요